In [2]:
import json, re, random, time
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from data import load_data, run_sql, same_result
from config import CONFIG, SEED, QUICK_RUN, set_seed
set_seed()

TEST_SIZE = 200
VAL_SIZE = 100
QUICK_TEST = 20
AUDIT_N = 60 
CACHE = Path("clean_data"); CACHE.mkdir(exist_ok=True) 

In [5]:
import warnings 
warnings.filterwarnings("ignore", category=UserWarning)
ds = load_data()
raw_train, raw_test = ds["train"], ds["test"]
len(raw_train),len(raw_test)

(100000, 5851)

In [6]:
def check_example(ex):
    sql=ex['sql'].strip()
    if not sql.lower().startswith(("select", "with")):
        return "not_select"
    rows, err = run_sql(ex["sql_context"], sql)
    if err is not None:
        e = err.lower()
        if "no such table" in e:    return "missing_table"
        if "no such column" in e:   return "missing_column"
        if "no such function" in e: return "unsupported_function"
        if "syntax error" in e:     return "syntax_dialect"
        if "timed out" in e:        return "timeout"
        if "setup error" in e:      return "bad_context"
        return "other_error"
    if not rows:
        return "empty_result"
    if len(rows) == 1 and len(rows[0]) == 1 and rows[0][0] in (0, None, ""):
        return "trivial_result"
    return "ok"

    


In [7]:
def run_checks(split, name):
    cache = CACHE / f"checks_{name}.json"
    if cache.exists():
        return json.load(open(cache))
    res = {}
    for ex in tqdm(split, desc=name):
        res[str(ex["id"])] = check_example(ex)
    json.dump(res, open(cache, "w"))
    return res

In [8]:
test_checks = run_checks(raw_test, "test")
train_checks = run_checks(raw_train, "train")

test:   0%|          | 0/5851 [00:00<?, ?it/s]

train:   0%|          | 0/100000 [00:00<?, ?it/s]